In [1]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))
print("✅ Ready for Day 2 - Prompt Engineering!")

✅ Ready for Day 2 - Prompt Engineering!


In [2]:
# ZERO SHOT - no examples given
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "Classify this review as POSITIVE or NEGATIVE: 'The food was amazing and the service was excellent!'"}
    ]
)
print(response.choices[0].message.content)

I would classify this review as: POSITIVE. The reviewer uses words like "amazing" and "excellent" to describe their experience, indicating a very positive opinion.


In [3]:
# FEW SHOT - giving examples to guide the model
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": """Classify reviews as POSITIVE or NEGATIVE. Reply with just one word.

Examples:
Review: "Great product, love it!" → POSITIVE
Review: "Terrible quality, broke immediately" → NEGATIVE
Review: "Best purchase ever!" → POSITIVE
Review: "Complete waste of money" → NEGATIVE

Now classify this:
Review: "The food was amazing and the service was excellent!" →"""}
    ]
)
print(response.choices[0].message.content)

POSITIVE


In [4]:
# WITHOUT Chain of Thought
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": "A store sells apples for $1.50 each. John buys 5 apples and pays with a $20 bill. He then buys 3 more apples. How much money does he have left?"}
    ]
)
print("WITHOUT CoT:")
print(response.choices[0].message.content)

WITHOUT CoT:
To determine how much money John has left, first, we need to calculate the total cost of the 5 apples he bought initially. 

5 apples * $1.50 per apple = $7.50

He paid with a $20 bill, so the change he got is:
$20 - $7.50 = $12.50

Then, he bought 3 more apples:
3 apples * $1.50 per apple = $4.50

Now, subtract the cost of the 3 additional apples from the change:
$12.50 - $4.50 = $8.00

So, John has $8.00 left.


In [5]:
# WITH Chain of Thought
response = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[
        {"role": "user", "content": """A store sells apples for $1.50 each. John buys 5 apples and pays with a $20 bill. He then buys 3 more apples. How much money does he have left?

Think through this step by step before giving your final answer."""}
    ]
)
print("WITH CoT:")
print(response.choices[0].message.content)

WITH CoT:
To find out how much money John has left, we need to calculate the total amount he spent on apples and subtract that from the $20 bill he initially paid with.

First, John buys 5 apples for $1.50 each. The total cost of these apples is:
5 apples * $1.50/apple = $7.50

He pays with a $20 bill, so the amount of money he has left after buying the first 5 apples is:
$20 - $7.50 = $12.50

Then, John buys 3 more apples for $1.50 each. The total cost of these apples is:
3 apples * $1.50/apple = $4.50

Now, we subtract the cost of the additional 3 apples from the amount of money he had left:
$12.50 - $4.50 = $8.00

Therefore, John has $8.00 left.


In [6]:
# Hard logic problem - see the difference clearly
hard_question = """I have 3 boxes. Box A has only apples. 
Box B has only oranges. Box C has both apples and oranges.
All boxes are INCORRECTLY labeled.
I pick one fruit from one box without looking inside.
I pick from the box labeled 'Both'.
I get an apple.
What is in each box?"""

# Without CoT
r1 = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": hard_question}]
)
print("WITHOUT CoT:")
print(r1.choices[0].message.content)
print("\n" + "="*50 + "\n")

# With CoT
r2 = client.chat.completions.create(
    model="llama-3.3-70b-versatile",
    messages=[{"role": "user", "content": hard_question + "\n\nThink through this carefully step by step."}]
)
print("WITH CoT:")
print(r2.choices[0].message.content)

WITHOUT CoT:
Since you picked an apple from the box labeled 'Both' and all labels are incorrect, this means the box labeled 'Both' actually contains only apples. 

So, Box A (labeled 'Apples') must contain only oranges because it can't have apples (already found in the 'Both' box), and it can't have both since the 'Both' box is actually the 'Apples' box. 

Box B (labeled 'Oranges') can't have only oranges because then the label would be correct. Since Box A has oranges, Box B must have both apples and oranges. 

Box C (labeled 'Both') has only apples, as you've already picked an apple from it.

So, the correct contents of each box are:
- Box A (labeled 'Apples'): Oranges
- Box B (labeled 'Oranges'): Both apples and oranges
- Box C (labeled 'Both'): Apples


WITH CoT:
To solve this problem, let's analyze the given information step by step.

1. **All boxes are INCORRECTLY labeled.** This means:
   - The box labeled 'Apples' does not contain only apples.
   - The box labeled 'Oranges' doe

In [7]:
def generate_content(topic):
    print(f"\n🚀 Generating content for: {topic}")
    print("=" * 60)
    
    # --- BLOG POST ---
    blog = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a professional blogger. Write engaging blog posts in markdown format with a title, introduction, 3 sections with headers, and a conclusion."},
            {"role": "user", "content": f"Write a blog post about: {topic}"}
        ],
        temperature=0.7,
        max_tokens=800
    )
    print("\n📝 BLOG POST:")
    print(blog.choices[0].message.content)
    
    # --- TWEET THREAD ---
    tweets = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a Twitter expert. Write viral tweet threads. Each tweet must be under 280 characters. Number them 1/5, 2/5 etc. Use emojis."},
            {"role": "user", "content": f"Write a 5 tweet thread about: {topic}"}
        ],
        temperature=0.9,
        max_tokens=400
    )
    print("\n🐦 TWEET THREAD:")
    print(tweets.choices[0].message.content)
    
    # --- LINKEDIN POST ---
    linkedin = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": "You are a LinkedIn influencer. Write professional posts with a strong hook first line, 3 key insights, and end with a question to drive comments."},
            {"role": "user", "content": f"Write a LinkedIn post about: {topic}"}
        ],
        temperature=0.6,
        max_tokens=400
    )
    print("\n💼 LINKEDIN POST:")
    print(linkedin.choices[0].message.content)

# Test it!
generate_content("The Future of AI in Healthcare")


🚀 Generating content for: The Future of AI in Healthcare

📝 BLOG POST:
# The Future of AI in Healthcare
The integration of Artificial Intelligence (AI) in healthcare has been a topic of interest for several years, with its potential to revolutionize the way medical professionals diagnose, treat, and manage patient care. As AI technology continues to advance, it's essential to explore the future of AI in healthcare and how it will impact the medical industry as a whole. In this blog post, we'll delve into the exciting developments and innovations that AI will bring to healthcare, and what we can expect in the years to come.

## Current Applications of AI in Healthcare
AI is already being used in various healthcare applications, including medical imaging, patient data analysis, and personalized medicine. For instance, AI-powered algorithms can analyze medical images such as X-rays and MRIs to help doctors diagnose diseases more accurately and quickly. Additionally, AI can analyze large 